# LeetCode 438: Find All Anagrams in a String

**Difficulty**: Medium  
**Topics**: Hash Table, String, Sliding Window  
**Link**: [LeetCode Problem](https://leetcode.com/problems/find-all-anagrams-in-a-string/)

---

## Problem Statement

Given two strings `s` and `p`, return an array of all the start indices of `p`'s anagrams in `s`. You may return the answer in **any order**.

An **anagram** is a word or phrase formed by rearranging the letters of a different word or phrase, typically using all the original letters exactly once.

### Examples

**Example 1:**
```
Input: s = "cbaebabacd", p = "abc"
Output: [0, 6]
Explanation:
The substring with start index = 0 is "cba", which is an anagram of "abc".
The substring with start index = 6 is "bac", which is an anagram of "abc".
```

**Example 2:**
```
Input: s = "abab", p = "ab"
Output: [0, 1, 2]
Explanation:
The substring with start index = 0 is "ab", which is an anagram of "ab".
The substring with start index = 1 is "ba", which is an anagram of "ab".
The substring with start index = 2 is "ab", which is an anagram of "ab".
```

### Constraints

- `1 <= s.length, p.length <= 3 * 10^4`
- `s` and `p` consist of lowercase English letters.

---

## Understanding the Problem

### What is an Anagram?

Two strings are anagrams if they contain the **same characters with the same frequencies**, regardless of order.

```
Examples:
"abc" and "bca" → Anagrams ✓
"abc" and "cab" → Anagrams ✓
"abc" and "def" → Not anagrams ✗
"aab" and "aba" → Anagrams ✓
"aab" and "aaa" → Not anagrams ✗
```

### Problem Breakdown

We need to:
1. Find all substrings of `s` with length equal to `p`
2. Check if each substring is an anagram of `p`
3. Return the starting indices of matching substrings

### Visual Example

```
s = "cbaebabacd"
p = "abc"

Step 1: Check all substrings of length 3

Index 0: "cba" → {c:1, b:1, a:1} = {a:1, b:1, c:1} ✓ Anagram!
Index 1: "bae" → {b:1, a:1, e:1} ✗ Not anagram (has 'e')
Index 2: "aeb" → {a:1, e:1, b:1} ✗ Not anagram (has 'e')
Index 3: "eba" → {e:1, b:1, a:1} ✗ Not anagram (has 'e')
Index 4: "bab" → {b:2, a:1} ✗ Not anagram (wrong counts)
Index 5: "aba" → {a:2, b:1} ✗ Not anagram (wrong counts)
Index 6: "bac" → {b:1, a:1, c:1} = {a:1, b:1, c:1} ✓ Anagram!
Index 7: "acd" → {a:1, c:1, d:1} ✗ Not anagram (has 'd')

Result: [0, 6]
```

### Key Insights

1. **Fixed window size**: All substrings have length `len(p)`
2. **Character frequency**: Anagrams have identical character counts
3. **Sliding window**: Can efficiently check all substrings
4. **Hash map comparison**: Compare frequency maps

---

## Approach 1: Brute Force with Sorting

### Algorithm

1. For each possible starting index in `s`:
   - Extract substring of length `len(p)`
   - Sort both `p` and substring
   - If sorted strings match, add index to result

### Complexity

- **Time**: O(n × m log m) where n = len(s), m = len(p)
  - n substrings to check
  - Each sort takes O(m log m)
- **Space**: O(m) for sorting

### Why Slow?

- Sorting is expensive for each substring
- Doesn't reuse information from previous windows
- Not optimal for large inputs

In [ ]:
def findAnagrams_bruteforce(s, p):
    """
    Brute force with sorting.
    Time: O(n × m log m)
    Space: O(m)
    """
    result = []
    p_len = len(p)
    s_len = len(s)
    
    if s_len < p_len:
        return result
    
    p_sorted = sorted(p)
    
    for i in range(s_len - p_len + 1):
        substring = s[i:i + p_len]
        if sorted(substring) == p_sorted:
            result.append(i)
    
    return result

# Test
print("Approach 1: Brute Force with Sorting\n")
s1, p1 = "cbaebabacd", "abc"
print(f"s = '{s1}', p = '{p1}'")
print(f"Result: {findAnagrams_bruteforce(s1, p1)}\n")

s2, p2 = "abab", "ab"
print(f"s = '{s2}', p = '{p2}'")
print(f"Result: {findAnagrams_bruteforce(s2, p2)}")

---

## Approach 2: Sliding Window with Hash Map

### Key Idea

Use a **sliding window** of size `len(p)` and maintain character frequency counts.

### Algorithm

1. Create frequency map for `p`
2. Create frequency map for first window in `s`
3. Compare maps, add index if match
4. Slide window:
   - Remove leftmost character
   - Add new rightmost character
   - Compare maps again
5. Repeat until end of `s`

### Visual Example

```
s = "cbaebabacd", p = "abc"
p_count = {a:1, b:1, c:1}

Window 0: [c b a] e b a b a c d
          -----
          {c:1, b:1, a:1} = p_count ✓ → Add 0

Window 1: c [b a e] b a b a c d
            -----
          Remove 'c', Add 'e'
          {b:1, a:1, e:1} ≠ p_count ✗

Window 2: c b [a e b] a b a c d
              -----
          Remove 'b', Add 'b'
          {a:1, e:1, b:1} ≠ p_count ✗

...

Window 6: c b a e b a [b a c] d
                      -----
          {b:1, a:1, c:1} = p_count ✓ → Add 6
```

### Complexity

- **Time**: O(n) where n = len(s)
  - Each character added/removed once
  - Map comparison is O(26) = O(1) for lowercase letters
- **Space**: O(1) - at most 26 characters in map

### Why Fast?

- Reuses information from previous window
- Only updates counts incrementally
- No sorting needed

In [ ]:
from collections import Counter

def findAnagrams_sliding_window(s, p):
    """
    Sliding window with hash map.
    Time: O(n)
    Space: O(1)
    """
    result = []
    p_len = len(p)
    s_len = len(s)
    
    if s_len < p_len:
        return result
    
    # Frequency map for p
    p_count = Counter(p)
    
    # Frequency map for first window
    window_count = Counter(s[:p_len])
    
    # Check first window
    if window_count == p_count:
        result.append(0)
    
    # Slide the window
    for i in range(p_len, s_len):
        # Add new character (right side)
        window_count[s[i]] += 1
        
        # Remove old character (left side)
        left_char = s[i - p_len]
        window_count[left_char] -= 1
        if window_count[left_char] == 0:
            del window_count[left_char]
        
        # Check if current window is anagram
        if window_count == p_count:
            result.append(i - p_len + 1)
    
    return result

# Test
print("Approach 2: Sliding Window with Hash Map\n")
s1, p1 = "cbaebabacd", "abc"
print(f"s = '{s1}', p = '{p1}'")
print(f"Result: {findAnagrams_sliding_window(s1, p1)}\n")

s2, p2 = "abab", "ab"
print(f"s = '{s2}', p = '{p2}'")
print(f"Result: {findAnagrams_sliding_window(s2, p2)}")

## Detailed Sliding Window Trace

In [ ]:
def findAnagrams_verbose(s, p):
    """
    Sliding window with detailed trace.
    """
    result = []
    p_len = len(p)
    s_len = len(s)
    
    if s_len < p_len:
        print("s is shorter than p, no anagrams possible")
        return result
    
    p_count = Counter(p)
    print(f"Target pattern p = '{p}'")
    print(f"Target frequency: {dict(p_count)}\n")
    print(f"String s = '{s}'")
    print(f"Window size: {p_len}\n")
    print("="*80)
    
    # First window
    window_count = Counter(s[:p_len])
    window_str = s[:p_len]
    
    print(f"\nInitial window [0:{p_len}]: '{window_str}'")
    print(f"Window frequency: {dict(window_count)}")
    print(f"Target frequency: {dict(p_count)}")
    
    if window_count == p_count:
        print(f"✓ MATCH! Add index 0")
        result.append(0)
    else:
        print(f"✗ No match")
    
    # Slide window
    for i in range(p_len, s_len):
        print(f"\n{'='*80}")
        print(f"\nSlide to position {i - p_len + 1}:")
        
        # Add new character
        new_char = s[i]
        print(f"  Add '{new_char}' (index {i})")
        window_count[new_char] += 1
        
        # Remove old character
        old_char = s[i - p_len]
        print(f"  Remove '{old_char}' (index {i - p_len})")
        window_count[old_char] -= 1
        if window_count[old_char] == 0:
            del window_count[old_char]
        
        # Current window
        start = i - p_len + 1
        window_str = s[start:i + 1]
        
        print(f"\n  Window [{start}:{i+1}]: '{window_str}'")
        print(f"  Window frequency: {dict(window_count)}")
        print(f"  Target frequency: {dict(p_count)}")
        
        if window_count == p_count:
            print(f"  ✓ MATCH! Add index {start}")
            result.append(start)
        else:
            print(f"  ✗ No match")
    
    print(f"\n{'='*80}")
    print(f"\nFinal result: {result}")
    return result

# Trace example 1
print("Example 1: Detailed Trace\n")
findAnagrams_verbose("cbaebabacd", "abc")

---

## Approach 3: Optimized Sliding Window with Match Count

### Optimization Idea

Instead of comparing entire hash maps, track the **number of characters that match**.

### Algorithm

1. Create frequency map for `p`
2. Track `matches` = number of characters with correct frequency
3. When `matches == len(p_count)`, we have an anagram
4. Update `matches` incrementally as we slide

### Key Insight

```
When adding/removing a character:
- If count becomes equal to target → matches++
- If count was equal to target → matches--
```

### Complexity

- **Time**: O(n) - same as approach 2
- **Space**: O(1) - at most 26 characters

### Advantage

- Avoids comparing entire maps
- O(1) check instead of O(26)
- Slightly faster in practice

In [ ]:
def findAnagrams_optimized(s, p):
    """
    Optimized sliding window with match count.
    Time: O(n)
    Space: O(1)
    """
    result = []
    p_len = len(p)
    s_len = len(s)
    
    if s_len < p_len:
        return result
    
    # Frequency maps
    p_count = {}
    window_count = {}
    
    # Initialize frequency maps
    for char in p:
        p_count[char] = p_count.get(char, 0) + 1
    
    # Build first window and count matches
    matches = 0
    for i in range(p_len):
        char = s[i]
        window_count[char] = window_count.get(char, 0) + 1
    
    # Count initial matches
    for char in p_count:
        if window_count.get(char, 0) == p_count[char]:
            matches += 1
    
    # Check first window
    if matches == len(p_count):
        result.append(0)
    
    # Slide window
    for i in range(p_len, s_len):
        # Add new character (right)
        right_char = s[i]
        if right_char in p_count:
            # Check if this addition creates a match
            if window_count.get(right_char, 0) == p_count[right_char]:
                matches -= 1  # Was matching, now exceeds
            window_count[right_char] = window_count.get(right_char, 0) + 1
            if window_count[right_char] == p_count[right_char]:
                matches += 1  # Now matches
        else:
            window_count[right_char] = window_count.get(right_char, 0) + 1
        
        # Remove old character (left)
        left_char = s[i - p_len]
        if left_char in p_count:
            # Check if this removal breaks a match
            if window_count[left_char] == p_count[left_char]:
                matches -= 1  # Was matching, now less
            window_count[left_char] -= 1
            if window_count[left_char] == p_count[left_char]:
                matches += 1  # Now matches
        else:
            window_count[left_char] -= 1
        
        # Check if all characters match
        if matches == len(p_count):
            result.append(i - p_len + 1)
    
    return result

# Test
print("Approach 3: Optimized Sliding Window\n")
s1, p1 = "cbaebabacd", "abc"
print(f"s = '{s1}', p = '{p1}'")
print(f"Result: {findAnagrams_optimized(s1, p1)}\n")

s2, p2 = "abab", "ab"
print(f"s = '{s2}', p = '{p2}'")
print(f"Result: {findAnagrams_optimized(s2, p2)}")

## Optimized Approach Trace with Match Count

In [ ]:
def findAnagrams_optimized_verbose(s, p):
    """
    Optimized approach with match count trace.
    """
    result = []
    p_len = len(p)
    s_len = len(s)
    
    if s_len < p_len:
        return result
    
    p_count = {}
    window_count = {}
    
    for char in p:
        p_count[char] = p_count.get(char, 0) + 1
    
    print(f"Target p = '{p}'")
    print(f"Target counts: {p_count}")
    print(f"Need {len(p_count)} characters to match\n")
    print("="*80)
    
    # Build first window
    for i in range(p_len):
        char = s[i]
        window_count[char] = window_count.get(char, 0) + 1
    
    # Count matches
    matches = 0
    for char in p_count:
        if window_count.get(char, 0) == p_count[char]:
            matches += 1
    
    print(f"\nInitial window: '{s[:p_len]}'")
    print(f"Window counts: {window_count}")
    print(f"Matches: {matches}/{len(p_count)}")
    
    if matches == len(p_count):
        print(f"✓ All characters match! Add index 0")
        result.append(0)
    else:
        print(f"✗ Not all characters match")
    
    # Slide window
    for i in range(p_len, s_len):
        print(f"\n{'='*80}")
        print(f"\nSlide to position {i - p_len + 1}:")
        
        # Add right
        right_char = s[i]
        print(f"  Add '{right_char}':")
        
        if right_char in p_count:
            old_count = window_count.get(right_char, 0)
            if old_count == p_count[right_char]:
                matches -= 1
                print(f"    Was matching ({old_count}), now will exceed → matches: {matches}")
            
            window_count[right_char] = old_count + 1
            new_count = window_count[right_char]
            
            if new_count == p_count[right_char]:
                matches += 1
                print(f"    Now matches target ({new_count}) → matches: {matches}")
            else:
                print(f"    Count: {new_count}, target: {p_count[right_char]}")
        else:
            window_count[right_char] = window_count.get(right_char, 0) + 1
            print(f"    Not in target pattern")
        
        # Remove left
        left_char = s[i - p_len]
        print(f"  Remove '{left_char}':")
        
        if left_char in p_count:
            old_count = window_count[left_char]
            if old_count == p_count[left_char]:
                matches -= 1
                print(f"    Was matching ({old_count}), now will be less → matches: {matches}")
            
            window_count[left_char] = old_count - 1
            new_count = window_count[left_char]
            
            if new_count == p_count[left_char]:
                matches += 1
                print(f"    Now matches target ({new_count}) → matches: {matches}")
            else:
                print(f"    Count: {new_count}, target: {p_count[left_char]}")
        else:
            window_count[left_char] -= 1
            print(f"    Not in target pattern")
        
        start = i - p_len + 1
        window_str = s[start:i + 1]
        print(f"\n  Window: '{window_str}'")
        print(f"  Matches: {matches}/{len(p_count)}")
        
        if matches == len(p_count):
            print(f"  ✓ All match! Add index {start}")
            result.append(start)
        else:
            print(f"  ✗ Not all match")
    
    print(f"\n{'='*80}")
    print(f"\nFinal result: {result}")
    return result

# Trace
print("Optimized Approach with Match Count Trace\n")
findAnagrams_optimized_verbose("cbaebabacd", "abc")

---

## Edge Cases and Special Scenarios

In [ ]:
print("Edge Cases Testing\n")
print("="*70)

# Edge Case 1: s shorter than p
print("\nCase 1: s shorter than p")
s, p = "ab", "abc"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Impossible to find anagram\n")

# Edge Case 2: s equals p
print("Case 2: s equals p")
s, p = "abc", "abc"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Entire string is anagram\n")

# Edge Case 3: Single character
print("Case 3: Single character")
s, p = "a", "a"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Single character match\n")

# Edge Case 4: No anagrams
print("Case 4: No anagrams")
s, p = "abcdefg", "xyz"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: No matching characters\n")

# Edge Case 5: All same character
print("Case 5: All same character")
s, p = "aaaa", "aa"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Multiple overlapping anagrams\n")

# Edge Case 6: Entire string is anagrams
print("Case 6: Entire string is anagrams")
s, p = "abab", "ab"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Every position is anagram\n")

# Edge Case 7: Duplicates in p
print("Case 7: Duplicates in p")
s, p = "aabbcc", "aab"
print(f"s = '{s}', p = '{p}'")
print(f"Result: {findAnagrams_sliding_window(s, p)}")
print("Explanation: Must match exact counts\n")

# Edge Case 8: Long string, short pattern
print("Case 8: Long string, short pattern")
s, p = "a" * 100 + "b", "a"
print(f"s = 'a'*100 + 'b', p = 'a'")
result = findAnagrams_sliding_window(s, p)
print(f"Result: {len(result)} anagrams found")
print(f"Indices: [0, 1, 2, ..., 99]")
print("Explanation: Every 'a' is an anagram")

---

## Performance Comparison

In [ ]:
import time
import random
import string

def generate_test_string(length):
    """Generate random lowercase string."""
    return ''.join(random.choices(string.ascii_lowercase[:5], k=length))

def benchmark(s, p):
    """Benchmark all approaches."""
    results = {}
    
    # Brute force
    start = time.time()
    result = findAnagrams_bruteforce(s, p)
    elapsed = time.time() - start
    results['Brute Force'] = (len(result), elapsed)
    
    # Sliding window
    start = time.time()
    result = findAnagrams_sliding_window(s, p)
    elapsed = time.time() - start
    results['Sliding Window'] = (len(result), elapsed)
    
    # Optimized
    start = time.time()
    result = findAnagrams_optimized(s, p)
    elapsed = time.time() - start
    results['Optimized'] = (len(result), elapsed)
    
    return results

# Benchmark
print("Performance Comparison\n")
print(f"{'String Length':<15} {'Approach':<20} {'Anagrams':<12} {'Time (seconds)'}")
print("="*70)

for length in [100, 500, 1000, 5000]:
    s = generate_test_string(length)
    p = "abc"
    
    results = benchmark(s, p)
    
    for approach, (count, elapsed) in results.items():
        print(f"{length:<15} {approach:<20} {count:<12} {elapsed:.6f}")
    print()

print("Observation:")
print("- Brute force is slowest (O(n × m log m))")
print("- Sliding window is fast (O(n))")
print("- Optimized is slightly faster (avoids full map comparison)")

---

## Comparison of Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Brute Force** | O(n × m log m) | O(m) | Simple, easy to understand | Very slow for large inputs |
| **Sliding Window** | O(n) | O(1) | Fast, reuses information | Need to compare maps |
| **Optimized** | O(n) | O(1) | Fastest, O(1) check | More complex logic |

Where:
- n = len(s)
- m = len(p)

### When to Use Each

- **Brute Force**: Understanding the problem, very small inputs
- **Sliding Window**: Interview (clear and efficient), production
- **Optimized**: Performance-critical applications

---

## Key Takeaways

### Anagram Definition

- **Same characters** with **same frequencies**
- Order doesn't matter
- Can check by:
  - Sorting both strings
  - Comparing frequency maps

### Sliding Window Pattern

1. **Fixed window size**: len(p)
2. **Initialize**: Build first window
3. **Slide**: Remove left, add right
4. **Check**: Compare frequencies
5. **Repeat**: Until end of string

### Sliding Window Template

```python
# Initialize window
window = build_first_window(s, window_size)

# Check first window
if is_valid(window):
    result.append(0)

# Slide window
for i in range(window_size, len(s)):
    # Remove left element
    remove(window, s[i - window_size])
    
    # Add right element
    add(window, s[i])
    
    # Check current window
    if is_valid(window):
        result.append(i - window_size + 1)
```

### Optimization Techniques

1. **Reuse information**: Don't rebuild from scratch
2. **Incremental updates**: Add/remove one element at a time
3. **Match counting**: Track matches instead of comparing maps
4. **Early termination**: Stop when impossible

### Common Mistakes

❌ **Rebuilding window each time**
```python
# WRONG - O(n × m)
for i in range(len(s) - len(p) + 1):
    window = Counter(s[i:i+len(p)])  # Rebuild every time!
```

✅ **Sliding window**
```python
# CORRECT - O(n)
window = Counter(s[:len(p)])
for i in range(len(p), len(s)):
    window[s[i]] += 1  # Add one
    window[s[i-len(p)]] -= 1  # Remove one
```

### Edge Cases to Remember

1. **s shorter than p**: Return empty list
2. **s equals p**: Check if anagram, return [0] or []
3. **Single character**: Simple case
4. **All same character**: Many overlapping anagrams
5. **No matches**: Return empty list
6. **Duplicates in p**: Must match exact counts

### Complexity Analysis

**Sliding Window:**
- **Time**: O(n) where n = len(s)
  - Each character added once: O(n)
  - Each character removed once: O(n)
  - Map comparison: O(26) = O(1) for lowercase letters
  - Total: O(n)
- **Space**: O(1)
  - At most 26 characters in map
  - Constant space regardless of input size

### Remember

🎯 **Sliding window** for fixed-size substrings  
🎯 **Frequency maps** to check anagrams  
🎯 **Incremental updates** for efficiency  
🎯 **Match counting** for optimization  
🎯 **O(n) time, O(1) space** is optimal  
🎯 **Reuse information** from previous window  

Master this problem and you'll understand:
- Sliding window technique
- Hash map for frequency counting
- Incremental updates
- String pattern matching